# Explore the invitations table

Runs after `mammo load_csv --path <file>`, before `features/build_train_set.py`.
One row per invitation; a person can appear several times.

In [1]:
import sys
print(sys.executable)      # should end in .../hackaton_mamography_ignorance/.venv/bin/python

/Users/alex/Projects_py/hackathons/prague_2026/hackaton_mamography_ignorance/.venv/bin/python


In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

from mammography.config import get_config
from mammography.io import read_table

config = get_config()
target = config["target"]

invitations = read_table("interim", "invitations", parse_dates=["invitation_date"])
invitations.shape

## 1. Shape of the table

Invitations vs people is the number that decides the train/test split: with
repeated people, a random split puts one invitation of a woman in train and
another in test, which leaks her behaviour across the split.

In [ ]:
print(f"invitations: {len(invitations)}")
print(f"people:      {invitations['person_id'].nunique()}")
print(f"duplicate invitation_id: {invitations['invitation_id'].duplicated().sum()}")
print(
    f"date range:  {invitations['invitation_date'].min().date()}"
    f" .. {invitations['invitation_date'].max().date()}"
)
print("\ninvitations per person:")
print(invitations["person_id"].value_counts().value_counts().sort_index().to_string())

## 2. The label, at three horizons

`screening_3m_flag`, `_6m_` and `_12m_` are the same outcome measured at
different windows. Picking one fixes the attendance window; the other two are
the answer at a different horizon and must never become features.

In [ ]:
horizons = ["screening_3m_flag", "screening_6m_flag", "screening_12m_flag"]

print("screened (True) share per horizon:")
print(invitations[horizons].mean().round(4).to_string())
print(f"\nchosen target: {target}")
print(f"ignore rate:   {1 - invitations[target].mean():.1%}")

print("\nconsistency - a 3m attender must also count at 6m and 12m:")
print(
    f"3m and not 6m:  "
    f"{(invitations['screening_3m_flag'] & ~invitations['screening_6m_flag']).sum()}"
)
print(
    f"6m and not 12m: "
    f"{(invitations['screening_6m_flag'] & ~invitations['screening_12m_flag']).sum()}"
)

## 3. Completeness

`months_since_last_screen` is empty wherever a woman was never screened, so its
null rate should track `never_screened_before_flag`. If it does, those nulls are
structural and mean "never", not "unknown".

In [ ]:
nulls = invitations.isna().mean().sort_values(ascending=False)
print(nulls[nulls > 0].round(4).to_string())

print(
    f"\nnever screened before:            "
    f"{invitations['never_screened_before_flag'].mean():.1%}"
)
print(
    f"months_since_last_screen missing: "
    f"{invitations['months_since_last_screen'].isna().mean():.1%}"
)

## 4. Ignore rate by candidate feature

Counts sit next to every rate - a 90% ignore rate on 11 invitations is noise,
and these tables are the ones that end up on slides.

In [ ]:
def ignore_rate_by(column: str, top: int | None = None) -> pd.DataFrame:
    grouped = invitations.groupby(column, dropna=False).agg(
        invitations=(target, "size"),
        ignore_rate=(target, lambda flags: 1 - flags.mean()),
    )
    if top:
        return grouped.sort_values("invitations", ascending=False).head(top)
    return grouped.sort_index()


for column in [
    "age_band",
    "invitation_sequence",
    "first_invitation",
    "never_screened_before_flag",
    "gp_registered",
    "gyn_registered_flag",
    "gyn_reg_duration_band",
]:
    print(f"\n=== {column} ===")
    print(ignore_rate_by(column).round(4).to_string())

In [ ]:
print(f"distinct territorial areas: {invitations['vzp_territorial_area'].nunique()}")
print("\n=== vzp_territorial_area, 15 largest ===")
print(ignore_rate_by("vzp_territorial_area", top=15).round(4).to_string())

## 5. Engagement with the health system

The strongest candidates in this table: a woman who never contacts a doctor is
unlikely to come for screening, and gynaecological prevention visits are the
closest proxy for already being in the screening habit.

In [ ]:
engagement = [
    "healthcare_contact_12m_cnt",
    "gp_prevention_24m_cnt",
    "gyn_prevention_24m_cnt",
    "other_screening_36m_cnt",
    "dental_prevention_24m_cnt",
    "prior_screen_count_5y",
    "prior_invitation_count",
    "insurance_duration_years",
]

print(invitations[engagement].describe().round(2).to_string())

In [ ]:
bands = pd.cut(
    invitations["healthcare_contact_12m_cnt"], [-1, 0, 2, 5, 10, 20, 1_000_000]
)
by_contact = invitations.groupby(bands, observed=True).agg(
    invitations=(target, "size"),
    ignore_rate=(target, lambda flags: 1 - flags.mean()),
)
print("ignore rate by healthcare contacts in 12 months:")
print(by_contact.round(4).to_string())

by_contact["ignore_rate"].plot(kind="bar")
plt.ylabel("ignore rate")
plt.xlabel("healthcare contacts in 12m")
plt.tight_layout()
plt.show()

## 6. Volume and ignore rate over time

Decides the time-based split, and exposes censoring: invitations near the end of
the file may not have had 12 months to resolve yet. If the ignore rate climbs
sharply in the last months, those rows are unresolved rather than ignored and
have to be dropped.

In [ ]:
monthly = (
    invitations.set_index("invitation_date")
    .resample("MS")
    .agg(
        invitations=(target, "size"),
        ignore_rate=(target, lambda flags: 1 - flags.mean()),
    )
)

figure, (upper, lower) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
monthly["invitations"].plot(ax=upper)
upper.set_ylabel("invitations")
monthly["ignore_rate"].plot(ax=lower)
lower.set_ylabel("ignore rate")
plt.tight_layout()
plt.show()

monthly.tail(12).round(4)